# 🛒 E-Commerce Customer Intelligence & Sales Analytics

## Phase 1 — Raw Data Audit

### Objective
Perform a comprehensive audit of the Olist Brazilian E-Commerce dataset before data cleaning and transformation.

### Tools
- Python
- Pandas
- NumPy

### Audit Areas
- Dataset structure
- Data types
- Missing values
- Duplicate records
- Unique identifiers
- Data quality
- Date ranges
- Table relationships

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_PATH = PROJECT_ROOT / "01_Raw_Data"

print("Project Root:", PROJECT_ROOT)
print("Raw Data Path:", RAW_DATA_PATH)
print("\nFiles found:")

for file in RAW_DATA_PATH.glob("*.csv"):
    print(" -", file.name)

Project Root: c:\Users\Dell\Desktop\Ecommerce-Customer-Intelligence
Raw Data Path: c:\Users\Dell\Desktop\Ecommerce-Customer-Intelligence\01_Raw_Data

Files found:
 - olist_customers_dataset.csv
 - olist_geolocation_dataset.csv
 - olist_orders_dataset.csv
 - olist_order_items_dataset.csv
 - olist_order_payments_dataset.csv
 - olist_order_reviews_dataset.csv
 - olist_products_dataset.csv
 - olist_sellers_dataset.csv
 - product_category_name_translation.csv


In [4]:
# Load all rows dataset

datasets = {}

for file in RAW_DATA_PATH.glob("*.csv"):
    dataset_name = file.stem.replace("olist_","").replace("_dataset","")

    datasets[dataset_name] = pd.read_csv(file)

    print(f"{dataset_name:20} -> {datasets[dataset_name].shape}")

customers            -> (99441, 5)
geolocation          -> (1000163, 5)
orders               -> (99441, 8)
order_items          -> (112650, 7)
order_payments       -> (103886, 5)
order_reviews        -> (99224, 7)
products             -> (32951, 9)
sellers              -> (3095, 4)
product_category_name_translation -> (71, 2)


In [8]:
# Dataset-level audit

audit_summary = []

for name , df in datasets.items():
    audit_summary.append({
        "Dataset" : name,
        "Raws":df.shape[0],
        "columns": df.shape[1],
        "Duplicate Rows": df.duplicated().sum(),
        "Missing cells": df.isna().sum()
.sum(),
"Memory (MB)": round(df.memory_usage(deep = True).sum()/1024**2 ,2) 
  })

audit_df = pd.DataFrame(audit_summary)

audit_df

,Dataset,Raws,columns,Duplicate Rows,Missing cells,Memory (MB)
0,customers,99441,5,0,0,11.03
1,geolocation,1000163,5,261831,0,50.12
2,orders,99441,8,0,4908,21.95
3,order_items,112650,7,0,0,18.37
4,order_payments,103886,5,0,0,8.11
5,order_reviews,99224,7,0,145903,17.84
6,products,32951,9,0,2448,3.73
7,sellers,3095,4,0,0,0.22
8,product_category_name_translation,71,2,0,0,0.00


## 2. Column-Level Data Quality Audit

This section investigates each column across all raw datasets.

We will examine:
- Data type
- Missing values
- Missing percentage
- Unique values
- Duplicate values
- Potential identifier columns

In [9]:
# Column-level data quality audit

column_audits = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        
        missing_count = df[column].isna().sum()
        total_rows = len(df)
        
        column_audits.append({
            "Dataset": dataset_name,
            "Column": column,
            "Data Type": str(df[column].dtype),
            "Rows": total_rows,
            "Missing": missing_count,
            "Missing %": round((missing_count / total_rows) * 100, 2),
            "Unique Values": df[column].nunique(dropna=True),
            "Unique %": round(
                (df[column].nunique(dropna=True) / total_rows) * 100, 2
            )
        })

column_audit_df = pd.DataFrame(column_audits)

column_audit_df

,Dataset,Column,Data Type,Rows,Missing,Missing %,Unique Values,Unique %
0,customers,customer_id,str,99441,0,0.00,99441,100.00
1,customers,customer_unique_id,str,99441,0,0.00,96096,96.64
2,customers,customer_zip_code_prefix,int64,99441,0,0.00,14994,15.08
3,customers,customer_city,str,99441,0,0.00,4119,4.14
4,customers,customer_state,str,99441,0,0.00,27,0.03
5,geolocation,geolocation_zip_code_prefix,int64,1000163,0,0.00,19015,1.90
6,geolocation,geolocation_lat,float64,1000163,0,0.00,717360,71.72
7,geolocation,geolocation_lng,float64,1000163,0,0.00,717613,71.75
8,geolocation,geolocation_city,str,1000163,0,0.00,8011,0.80
9,geolocation,geolocation_state,str,1000163,0,0.00,27,0.00


In [10]:
missing_columns = (
    column_audit_df[
        column_audit_df["Missing"] > 0
    ]
    .sort_values(
        by="Missing %",
        ascending=False
    )
)

missing_columns

,Dataset,Column,Data Type,Rows,Missing,Missing %,Unique Values,Unique %
33,order_reviews,review_comment_title,str,99224,87656,88.34,4527,4.56
34,order_reviews,review_comment_message,str,99224,58247,58.70,36159,36.44
16,orders,order_delivered_customer_date,str,99441,2965,2.98,95664,96.20
39,products,product_name_lenght,float64,32951,610,1.85,66,0.20
38,products,product_category_name,str,32951,610,1.85,73,0.22
40,products,product_description_lenght,float64,32951,610,1.85,2960,8.98
41,products,product_photos_qty,float64,32951,610,1.85,19,0.06
15,orders,order_delivered_carrier_date,str,99441,1783,1.79,81018,81.47
14,orders,order_approved_at,str,99441,160,0.16,90733,91.24
42,products,product_weight_g,float64,32951,2,0.01,2204,6.69


In [11]:
geo = datasets["geolocation"]

print("Total rows:", len(geo))
print("Exact duplicate rows:", geo.duplicated().sum())

print("\nDuplicate percentage:")
print(
    round(
        geo.duplicated().sum() / len(geo) * 100,
        2
    ),
    "%"
)

Total rows: 1000163
Exact duplicate rows: 261831

Duplicate percentage:
26.18 %


In [12]:
geo.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [13]:
geo_zip_summary = (
    geo.groupby("geolocation_zip_code_prefix")
       .agg(
           records=("geolocation_zip_code_prefix", "size"),
           lat_unique=("geolocation_lat", "nunique"),
           lng_unique=("geolocation_lng", "nunique"),
           city_unique=("geolocation_city", "nunique"),
           state_unique=("geolocation_state", "nunique")
       )
       .sort_values("records", ascending=False)
)

geo_zip_summary.head(10)

,records,lat_unique,lng_unique,city_unique,state_unique
geolocation_zip_code_prefix,,,,,
24220,1146,410,411,2,1
24230,1102,327,327,2,1
38400,965,746,745,2,1
35500,907,726,726,2,1
11680,879,726,726,1,1
22631,832,141,142,1,1
30140,810,379,380,1,1
11740,788,666,666,2,1
38408,773,600,599,2,1


## 3. Relational Structure & Key Validation

The Olist dataset consists of multiple related tables.

This section identifies:
- Primary keys
- Foreign keys
- Cardinality
- Referential integrity
- Potential orphan records

In [14]:
# Display all columns for every dataset

for name, df in datasets.items():
    print(f"\n{'=' * 70}")
    print(f"{name.upper()}")
    print("=" * 70)
    print(list(df.columns))


CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

GEOLOCATION
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

ORDER_PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

ORDER_REVIEWS
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

PRODUCTS
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_heigh

In [15]:
# Candidate primary-key validation

candidate_keys = {
    "customers": ["customer_id", "customer_unique_id"],
    "orders": ["order_id"],
    "order_items": ["order_id"],
    "order_payments": ["order_id"],
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "geolocation": ["geolocation_zip_code_prefix"],
}

for dataset_name, columns in candidate_keys.items():
    
    df = datasets[dataset_name]
    
    print(f"\n{'=' * 70}")
    print(dataset_name.upper())
    print("=" * 70)
    
    for column in columns:
        if column in df.columns:
            print(
                f"{column:35} "
                f"Rows: {len(df):8,} | "
                f"Unique: {df[column].nunique():8,} | "
                f"Duplicates: {df[column].duplicated().sum():8,}"
            )


CUSTOMERS
customer_id                         Rows:   99,441 | Unique:   99,441 | Duplicates:        0
customer_unique_id                  Rows:   99,441 | Unique:   96,096 | Duplicates:    3,345

ORDERS
order_id                            Rows:   99,441 | Unique:   99,441 | Duplicates:        0

ORDER_ITEMS
order_id                            Rows:  112,650 | Unique:   98,666 | Duplicates:   13,984

ORDER_PAYMENTS
order_id                            Rows:  103,886 | Unique:   99,440 | Duplicates:    4,446

ORDER_REVIEWS
review_id                           Rows:   99,224 | Unique:   98,410 | Duplicates:      814

PRODUCTS
product_id                          Rows:   32,951 | Unique:   32,951 | Duplicates:        0

SELLERS
seller_id                           Rows:    3,095 | Unique:    3,095 | Duplicates:        0

GEOLOCATION
geolocation_zip_code_prefix         Rows: 1,000,163 | Unique:   19,015 | Duplicates:  981,148


In [16]:
# Referential integrity checks

relationships = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id"),
]

relationship_results = []

for child_table, child_key, parent_table, parent_key in relationships:
    
    child = datasets[child_table]
    parent = datasets[parent_table]
    
    parent_keys = set(parent[parent_key].dropna().unique())
    
    orphan_mask = (
        child[child_key].notna() &
        ~child[child_key].isin(parent_keys)
    )
    
    orphan_count = orphan_mask.sum()
    
    relationship_results.append({
        "Child Table": child_table,
        "Child Key": child_key,
        "Parent Table": parent_table,
        "Parent Key": parent_key,
        "Child Rows": len(child),
        "Orphan Rows": orphan_count,
        "Orphan %": round(
            orphan_count / len(child) * 100, 4
        )
    })

relationship_df = pd.DataFrame(relationship_results)

relationship_df

,Child Table,Child Key,Parent Table,Parent Key,Child Rows,Orphan Rows,Orphan %
0,orders,customer_id,customers,customer_id,99441,0,0.0
1,order_items,order_id,orders,order_id,112650,0,0.0
2,order_items,product_id,products,product_id,112650,0,0.0
3,order_items,seller_id,sellers,seller_id,112650,0,0.0
4,order_payments,order_id,orders,order_id,103886,0,0.0
5,order_reviews,order_id,orders,order_id,99224,0,0.0


## 4. Official Data Model

### Core Relationships

- Customers → Orders: 1-to-many
- Orders → Order Items: 1-to-many
- Products → Order Items: 1-to-many
- Sellers → Order Items: 1-to-many
- Orders → Payments: 1-to-many
- Orders → Reviews: 1-to-many

### Referential Integrity

All tested foreign-key relationships contain zero orphan records.

This confirms that the raw Olist dataset has a consistent relational structure suitable for SQL database modeling and downstream BI analysis.

In [17]:
# Primary-key uniqueness audit

primary_key_candidates = {
    "customers": ["customer_id", "customer_unique_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "order_reviews": ["review_id"],
}

pk_results = []

for table, columns in primary_key_candidates.items():
    df = datasets[table]

    for column in columns:
        pk_results.append({
            "Table": table,
            "Column": column,
            "Rows": len(df),
            "Unique Values": df[column].nunique(),
            "Null Values": df[column].isna().sum(),
            "Duplicate Values": df[column].duplicated().sum(),
            "Is Unique": (
                df[column].nunique(dropna=False) == len(df)
            )
        })

pk_df = pd.DataFrame(pk_results)

pk_df

,Table,Column,Rows,Unique Values,Null Values,Duplicate Values,Is Unique
0,customers,customer_id,99441,99441,0,0,True
1,customers,customer_unique_id,99441,96096,0,3345,False
2,orders,order_id,99441,99441,0,0,True
3,products,product_id,32951,32951,0,0,True
4,sellers,seller_id,3095,3095,0,0,True
5,order_reviews,review_id,99224,98410,0,814,False


In [18]:
# Composite key investigation for order_reviews

reviews = datasets["order_reviews"]

review_key_check = {
    "Total Rows": len(reviews),
    "Unique review_id": reviews["review_id"].nunique(),
    "Unique order_id": reviews["order_id"].nunique(),
    "Unique review_id + order_id": (
        reviews[["review_id", "order_id"]]
        .drop_duplicates()
        .shape[0]
    ),
    "Duplicate review_id + order_id": (
        reviews.duplicated(
            subset=["review_id", "order_id"]
        ).sum()
    )
}

review_key_check

{'Total Rows': 99224,
 'Unique review_id': 98410,
 'Unique order_id': 98673,
 'Unique review_id + order_id': 99224,
 'Duplicate review_id + order_id': np.int64(0)}

In [19]:
reviews[
    reviews.duplicated(
        subset=["review_id", "order_id"],
        keep=False
    )
].sort_values(["review_id", "order_id"]).head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [20]:
# Data type audit

dtype_results = []

for table_name, df in datasets.items():
    for column in df.columns:
        dtype_results.append({
            "Table": table_name,
            "Column": column,
            "Data Type": str(df[column].dtype),
            "Null Values": df[column].isna().sum(),
            "Unique Values": df[column].nunique()
        })

dtype_df = pd.DataFrame(dtype_results)

dtype_df

,Table,Column,Data Type,Null Values,Unique Values
0,customers,customer_id,str,0,99441
1,customers,customer_unique_id,str,0,96096
2,customers,customer_zip_code_prefix,int64,0,14994
3,customers,customer_city,str,0,4119
4,customers,customer_state,str,0,27
5,geolocation,geolocation_zip_code_prefix,int64,0,19015
6,geolocation,geolocation_lat,float64,0,717360
7,geolocation,geolocation_lng,float64,0,717613
8,geolocation,geolocation_city,str,0,8011
9,geolocation,geolocation_state,str,0,27


In [21]:
# Show columns that look like dates but are currently stored as text

date_like_columns = dtype_df[
    dtype_df["Column"].str.contains(
        "date|timestamp|at",
        case=False,
        regex=True
    )
]

date_like_columns

,Table,Column,Data Type,Null Values,Unique Values
4,customers,customer_state,str,0,27
5,geolocation,geolocation_zip_code_prefix,int64,0,19015
6,geolocation,geolocation_lat,float64,0,717360
7,geolocation,geolocation_lng,float64,0,717613
8,geolocation,geolocation_city,str,0,8011
9,geolocation,geolocation_state,str,0,27
12,orders,order_status,str,0,8
13,orders,order_purchase_timestamp,str,0,98875
14,orders,order_approved_at,str,160,90733
15,orders,order_delivered_carrier_date,str,1783,81018


In [22]:
# Numeric columns and their basic statistics

numeric_summary = {}

for table_name, df in datasets.items():
    numeric_cols = df.select_dtypes(
        include=["number"]
    ).columns.tolist()

    if numeric_cols:
        numeric_summary[table_name] = df[numeric_cols].describe().T

numeric_summary

{'customers':                             count          mean           std     min  \
 customer_zip_code_prefix  99441.0  35137.474583  29797.938996  1003.0   
 
                               25%      50%      75%      max  
 customer_zip_code_prefix  11347.0  24416.0  58900.0  99990.0  ,
 'geolocation':                                  count          mean           std  \
 geolocation_zip_code_prefix  1000163.0  36574.166466  30549.335710   
 geolocation_lat              1000163.0    -21.176153      5.715866   
 geolocation_lng              1000163.0    -46.390541      4.269748   
 
                                      min           25%           50%  \
 geolocation_zip_code_prefix  1001.000000  11075.000000  26530.000000   
 geolocation_lat               -36.605374    -23.603546    -22.919377   
 geolocation_lng              -101.466766    -48.573172    -46.637879   
 
                                       75%           max  
 geolocation_zip_code_prefix  63504.000000  99990.0000

In [24]:
# Date/time conversion audit
# We are testing conversion only — RAW DATA is NOT being modified.

date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

date_audit = []

for table_name, columns in date_columns.items():
    df = datasets[table_name]

    for column in columns:
        converted = pd.to_datetime(
            df[column],
            errors="coerce"
        )

        date_audit.append({
            "Table": table_name,
            "Column": column,
            "Original Nulls": df[column].isna().sum(),
            "Conversion Failures": (
                converted.isna() & df[column].notna()
            ).sum(),
            "Min Date": converted.min(),
            "Max Date": converted.max()
        })

date_audit_df = pd.DataFrame(date_audit)

date_audit_df

,Table,Column,Original Nulls,Conversion Failures,Min Date,Max Date
0,orders,order_purchase_timestamp,0,0,2016-09-04 21:15:19,2018-10-17 17:30:18
1,orders,order_approved_at,160,0,2016-09-15 12:16:38,2018-09-03 17:40:06
2,orders,order_delivered_carrier_date,1783,0,2016-10-08 10:34:01,2018-09-11 19:48:28
3,orders,order_delivered_customer_date,2965,0,2016-10-11 13:46:32,2018-10-17 13:22:46
4,orders,order_estimated_delivery_date,0,0,2016-09-30 00:00:00,2018-11-12 00:00:00
5,order_items,shipping_limit_date,0,0,2016-09-19 00:15:34,2020-04-09 22:35:08
6,order_reviews,review_creation_date,0,0,2016-10-02 00:00:00,2018-08-31 00:00:00
7,order_reviews,review_answer_timestamp,0,0,2016-10-07 18:32:28,2018-10-29 12:27:35


In [25]:
# Geolocation coordinate validation

geo = datasets["geolocation"]

geo_coordinate_check = {
    "Total Rows": len(geo),

    "Invalid Latitude": (
        (geo["geolocation_lat"] < -90) |
        (geo["geolocation_lat"] > 90)
    ).sum(),

    "Invalid Longitude": (
        (geo["geolocation_lng"] < -180) |
        (geo["geolocation_lng"] > 180)
    ).sum(),

    "Latitude outside Brazil-like range": (
        (geo["geolocation_lat"] > 6) |
        (geo["geolocation_lat"] < -34)
    ).sum(),

    "Longitude outside Brazil-like range": (
        (geo["geolocation_lng"] > -34) |
        (geo["geolocation_lng"] < -74)
    ).sum()
}

geo_coordinate_check

{'Total Rows': 1000163,
 'Invalid Latitude': np.int64(0),
 'Invalid Longitude': np.int64(0),
 'Latitude outside Brazil-like range': np.int64(31),
 'Longitude outside Brazil-like range': np.int64(37)}

In [23]:
orders = datasets["orders"]

status_delivery_check = pd.crosstab(
    orders["order_status"],
    orders["order_delivered_customer_date"].isna()
)

status_delivery_check

order_delivered_customer_date,False,True
order_status,,
approved,0,2
canceled,6,619
created,0,5
delivered,96470,8
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,0,609


In [26]:
# Delivered orders missing customer delivery date

delivered_missing_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

delivered_missing_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [27]:
# Check all timestamps for these 8 orders

delivered_missing_date[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [29]:
geo_outliers = geo[
    (geo["geolocation_lat"] > 6) |
    (geo["geolocation_lat"] < -34) |
    (geo["geolocation_lng"] > -34) |
    (geo["geolocation_lng"] < -74)
]

geo_outliers

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


In [30]:
geo_outliers[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
].sort_values(
    ["geolocation_state", "geolocation_city"]
)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
727755,57319,45.065933,9.341528,pau d'arco,AL
697048,46560,38.991963,-4.947823,ibiajara,BA
695375,45936,-34.622400,-58.901888,itabata,BA
695377,45936,38.323939,-6.775035,itabatan,BA
698466,47310,38.268205,-7.803886,santana do sobrado,BA
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
585260,35179,25.995245,-98.078533,santana do paraiso,MG
585242,35179,25.995203,-98.078544,santana do paraíso,MG
822529,78131,38.816816,-9.394625,varzea grande,MT


In [31]:
items = datasets["order_items"]

items[
    items["shipping_limit_date"].str[:4].astype(int) > 2018
][
    [
        "order_id",
        "shipping_limit_date",
        "price",
        "freight_value"
    ]
].sort_values("shipping_limit_date").head(30)

,order_id,shipping_limit_date,price,freight_value
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,2020-02-03 20:23:22,75.99,14.70
8643,13bdf405f961a6deec817d817f5c6624,2020-02-05 03:30:51,69.99,14.66
85729,c2bb89b5c1dd978d507284be78a04cb2,2020-04-09 22:35:08,99.99,61.44
85730,c2bb89b5c1dd978d507284be78a04cb2,2020-04-09 22:35:08,99.99,61.44
